In [ ]:
# This notebook file was executed via Colab
# verification to make sure GPU is available for training
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


In [3]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes

In [1]:
# If Colab requests a restart, restart the session before continuing.
import torch
from huggingface_hub import notebook_login

assert torch.cuda.is_available(), "Select a GPU runtime first."

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    "GB",
)

notebook_login()

GPU: Tesla T4
GPU memory: 14.6 GB


In [ ]:
# Upload the three JSONL files and connect Drive
from google.colab import files, drive
from pathlib import Path
from datasets import load_dataset
import json
import shutil
import importlib.metadata

uploaded = files.upload()

required_files = ["train.jsonl", "validation.jsonl", "test.jsonl"]
assert all(name in uploaded for name in required_files)

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/Agent1/llama32_retail_pilot_v1"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

for name in required_files:
    shutil.copy2(name, RUN_DIR / name)

dataset = load_dataset(
    "json",
    data_files={
        "train": "train.jsonl",
        "validation": "validation.jsonl",
        "test": "test.jsonl",
    },
)

print(dataset)

versions = {
    package: importlib.metadata.version(package)
    for package in [
        "torch", "transformers", "datasets",
        "accelerate", "peft", "trl", "bitsandbytes"
    ]
}
(RUN_DIR / "versions.json").write_text(
    json.dumps(versions, indent=2),
    encoding="utf-8",
)

In [ ]:
# If data was uploaded already
from google.colab import drive
from pathlib import Path
from datasets import load_dataset
import json

drive.mount("/content/drive")

RUN_DIR = Path(
    "/content/drive/MyDrive/Agent1/llama32_retail_pilot_v1"
)

data_files = {
    split: RUN_DIR / f"{split}.jsonl"
    for split in ["train", "validation", "test"]
}

missing = [
    str(path)
    for path in data_files.values()
    if not path.is_file()
]

if missing:
    raise FileNotFoundError(f"Missing files in Drive: {missing}")

dataset = load_dataset(
    "json",
    data_files={
        split: str(path)
        for split, path in data_files.items()
    },
)

print(dataset)

In [4]:
# Load the model in 4-bit precision
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

set_seed(42)

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
USE_BF16 = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization,
    device_map={"": 0},
    torch_dtype=COMPUTE_DTYPE,
)

model.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [13]:
# Generate a small baseline before training
preview_examples = list(
    dataset["validation"].shuffle(seed=42).select(
        range(min(12, len(dataset["validation"])))
    )
)


def generate_answer(current_model, prompt):
    current_model.eval()

    inputs = tokenizer.apply_chat_template(
        prompt,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.inference_mode():
        output = current_model.generate(
            **inputs,
            max_new_tokens=384,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


comparisons = []

for example in preview_examples:
    comparisons.append({
        "intent": example["intent"],
        "prompt": example["prompt"],
        "reference": example["completion"][0]["content"],
        "before": generate_answer(model, example["prompt"]),
    })

(RUN_DIR / "baseline.json").write_text(
    json.dumps(comparisons, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print("USER:", comparisons[0]["prompt"][-1]["content"])
print("\nBEFORE:", comparisons[0]["before"])

In [5]:
# if baseline was saved already
comparisons = json.loads(
    (RUN_DIR / "baseline.json").read_text(encoding="utf-8")
)

In [6]:
# Prepare the training and validation examples
# For this first run, use a 1,024-token limit. Exclude longer examples instead of silently cutting off their target answers.
MAX_LENGTH = 1024


def fits_length(example):
    tokens = tokenizer.apply_chat_template(
        example["prompt"] + example["completion"],
        tokenize=True,
        add_generation_prompt=False,
    )
    return len(tokens) <= MAX_LENGTH


training_data = dataset["train"].filter(fits_length)
validation_data = dataset["validation"].filter(fits_length)

print("Training rows:", len(training_data))
print("Validation rows:", len(validation_data))

assert len(training_data) > 0 and len(validation_data) > 0

training_data = training_data.select_columns(
    ["prompt", "completion"]
).shuffle(seed=42)

validation_data = validation_data.select_columns(
    ["prompt", "completion"]
)

Filter:   0%|          | 0/1090 [00:00<?, ? examples/s]

Filter:   0%|          | 0/134 [00:00<?, ? examples/s]

Training rows: 1090
Validation rows: 134


In [8]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
# Attach LoRA adapters and train for one epoch
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from trl import SFTConfig, SFTTrainer

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.config.use_cache = False
model.print_trainable_parameters()

training_args = SFTConfig(
    output_dir=str(RUN_DIR / "checkpoints"),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    warmup_steps=0.05,
    lr_scheduler_type="cosine",
    max_length=MAX_LENGTH,
    completion_only_loss=True,
    packing=False,
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=training_data,
    eval_dataset=validation_data,
    processing_class=tokenizer,
)

baseline_metrics = trainer.evaluate()
print("Initial validation metrics:", baseline_metrics)

train_result = trainer.train()

final_metrics = trainer.evaluate()
print("Final validation metrics:", final_metrics)

In [8]:
# train with other approach as we have only 7.5 compute capability
import torch
import trl.kernels.chunked_logprob as chunked_logprob

print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))

# Workaround for the unsupported optimized matrix multiplication.
assert hasattr(chunked_logprob, "_MM_OUT_DTYPE")
chunked_logprob._MM_OUT_DTYPE = False

# Clear any gradients left by the failed training step.
trainer.model.zero_grad(set_to_none=True)

GPU: Tesla T4
Compute capability: (7, 5)


In [9]:
train_result = trainer.train()

final_metrics = trainer.evaluate()
print("Final validation metrics:", final_metrics)

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,0.617589,0.630584,0.643469,0.799573,428500.000000


Training Loss,Validation Loss,Epoch,Entropy,Mean Token Accuracy,Num Tokens
0.617589,0.630584,1,0.643469,0.799573,428500.000000


Final validation metrics: {'eval_loss': 0.6305841207504272, 'eval_entropy': 0.6434688138872829, 'eval_mean_token_accuracy': 0.7995727293526949, 'eval_num_tokens': 428500.0}


In [11]:
# Save the adapter and compare responses
adapter_dir = RUN_DIR / "adapter"

trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

print("\nSaved adapter:", adapter_dir)


Saved adapter: /content/drive/MyDrive/Agent1/llama32_retail_pilot_v1/adapter


In [14]:
(RUN_DIR / "metrics.json").write_text(
    json.dumps(
        {
            "before": baseline_metrics,
            "training": train_result.metrics,
            "after": final_metrics,
        },
        indent=2,
    ),
    encoding="utf-8",
)

trained_model = trainer.model
trained_model.gradient_checkpointing_disable()
trained_model.config.use_cache = True

for item in comparisons:
    item["after"] = generate_answer(trained_model, item["prompt"])

(RUN_DIR / "before_after.json").write_text(
    json.dumps(comparisons, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

for item in comparisons[:3]:
    print("\nUSER:", item["prompt"][-1]["content"])
    print("\nBEFORE:", item["before"])
    print("\nAFTER:", item["after"])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



USER: I want to check the fucking status of my replacement product, can I get some help?

BEFORE: I'd be happy to help you with the status of your replacement product. 

To assist you better, could you please provide me with your order number or the order ID associated with the replacement product you're inquiring about? This will allow me to look into your order and provide you with the most up-to-date information.

If you're unable to find the order number, you can also try checking your order history or your account settings on our website. If you need any assistance with that, I'd be more than happy to guide you through the process.

Additionally, if you'd like to speak with someone directly, you can also reach out to us through our support email or give us a call at +1-202-555-0147. Our support hours are Monday through Friday, 9:00 AM to 5:00 PM Eastern Time.

AFTER: I genuinely apologize for any frustration you may have experienced with your replacement product. I'm here to assi

In [15]:
print("Before training:", baseline_metrics)
print("After training:", final_metrics)

Before training: {'eval_loss': 1.2845946550369263, 'eval_entropy': 0.7340091996228517, 'eval_mean_token_accuracy': 0.6654575729548041, 'eval_num_tokens': 0.0}
After training: {'eval_loss': 0.6305841207504272, 'eval_entropy': 0.6434688138872829, 'eval_mean_token_accuracy': 0.7995727293526949, 'eval_num_tokens': 428500.0}


In [ ]:
import sys
import importlib.metadata as metadata
from pathlib import Path

print("Python:", sys.version)
print("Interpreter:", sys.executable)

for package in ["torch", "transformers", "peft", "accelerate", "bitsandbytes"]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

try:
    import torch

    print("\nCUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA version:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("Compute capability:", torch.cuda.get_device_capability(0))
except ImportError:
    print("\nPyTorch is not installed.")

relative_path = Path("models/llama32_retail_pilot_v1/adapter")

adapter_dir = next(
    (
        folder / relative_path
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / relative_path / "adapter_config.json").is_file()
    ),
    None,
)

print("\nAdapter directory:", adapter_dir)

if adapter_dir:
    print("Files:", [path.name for path in adapter_dir.iterdir()])